In [3]:
!pip install tensorflow opencv-python matplotlib kaggle

In [4]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle API Configured!")

Kaggle API Configured!


In [5]:
!kaggle datasets download -d gti-upm/leapgestrecog

Dataset URL: https://www.kaggle.com/datasets/gti-upm/leapgestrecog
License(s): CC-BY-NC-SA-4.0
100% 2.13G/2.13G [00:23<00:00, 96.1MB/s]



In [6]:
import zipfile

with zipfile.ZipFile("/content/leapgestrecog.zip", "r") as zip_ref:
    zip_ref.extractall("/content/data")

print("Dataset Extracted!")

Dataset Extracted!


In [7]:
import os

print(os.listdir("/content/data"))

['leapgestrecog', 'leapGestRecog']


In [8]:
import cv2
import numpy as np
import os

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [9]:
X = []
y = []

data_path = "/content/data/leapGestRecog"

labels = os.listdir(data_path)

for label in labels:
    folder = os.path.join(data_path, label)

    for person in os.listdir(folder):

        person_folder = os.path.join(folder, person)

        for img in os.listdir(person_folder):

            img_path = os.path.join(person_folder, img)

            image = cv2.imread(img_path)

            image = cv2.resize(image,(64,64))

            X.append(image)

            y.append(int(label))

X = np.array(X)
y = np.array(y)

print(X.shape)

(20000, 64, 64, 3)


In [10]:
X = X / 255.0

In [11]:
y = to_categorical(y)

In [12]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense

model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(64,64,3)))
model.add(MaxPooling2D())

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D())

model.add(Flatten())

model.add(Dense(128,activation='relu'))

model.add(Dense(y.shape[1],activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test,y_test)
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 94s 184ms/step - accuracy: 0.9252 - loss: 0.2273 - val_accuracy: 0.9885 - val_loss: 0.0273
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 95s 191ms/step - accuracy: 0.9911 - loss: 0.0221 - val_accuracy: 0.9877 - val_loss: 0.0335
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 92s 184ms/step - accuracy: 0.9923 - loss: 0.0168 - val_accuracy: 0.9875 - val_loss: 0.0285
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 140s 181ms/step - accuracy: 0.9927 - loss: 0.0145 - val_accuracy: 0.9935 - val_loss: 0.0145
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 95s 190ms/step - accuracy: 0.9918 - loss: 0.0148 - val_accuracy: 0.9898 - val_loss: 0.0170


In [16]:
loss,accuracy = model.evaluate(X_test,y_test)

print("Accuracy:",accuracy)

125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.9898 - loss: 0.0170
Accuracy: 0.9897500276565552


In [17]:
model.save("hand_gesture_model.h5")